# A3-M2 · IVF
교안의 같은 번호 단원과 함께 봅니다.

## 05-1. 준비 — 점 12개

In [1]:
import faiss
import numpy as np

x = np.array([[1, 1], [1, 2], [2, 1], [2, 2],      # 0~3: 왼쪽 아래 묶음
              [8, 8], [8, 9], [9, 8], [9, 9],      # 4~7: 오른쪽 위 묶음
              [1, 8], [1, 9], [2, 8], [2, 9]],     # 8~11: 왼쪽 위 묶음
             dtype=np.float32)
print(x.shape)

(12, 2)


## 05-2. IVF 만들기 — train이 하는 일

In [2]:
d, nlist = 2, 3
quantizer = faiss.IndexFlatL2(d)                       # 중심점을 담을 인덱스
ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)
print("학습됨?", ivf.is_trained)

try:
    ivf.add(x)                                         # 학습 전에 add 하면
except RuntimeError as e:
    print("오류: 학습 전에는 add 할 수 없음")

ivf.cp.min_points_per_centroid = 1                     # 작은 예제라 경고 끄기
ivf.train(x)                                           # 내부에서 k-means 실행
print("학습됨?", ivf.is_trained, "| 중심점 수:", quantizer.ntotal)
ivf.add(x)
print("ntotal:", ivf.ntotal, "| nlist:", ivf.nlist, "| nprobe:", ivf.nprobe)

학습됨? False
오류: 학습 전에는 add 할 수 없음
학습됨? True | 중심점 수: 3
ntotal: 12 | nlist: 3 | nprobe: 1


## 05-3. 목록(inverted list) 들여다보기 — add가 하는 일

In [3]:
centroids = quantizer.reconstruct_n(0, nlist)
print("중심점:\n", centroids)

def list_ids(c):                                       # c번 목록에 든 벡터 번호
    c = int(c)                                         # numpy 정수 → 파이썬 정수 (SWIG 함수가 요구)
    size = ivf.invlists.list_size(c)
    return faiss.rev_swig_ptr(ivf.invlists.get_ids(c), size).copy()

for c in range(nlist):
    print(f"목록 {c}: {ivf.invlists.list_size(c)}개 → 번호 {list_ids(c)}")

_, owner = quantizer.search(x, 1)                      # add는 점마다 이 계산을 해서 목록을 정한다
print("점마다 들어간 목록:", owner.ravel())

중심점:
 [[8.666667 8.666667]
 [1.5      5.      ]
 [8.       8.      ]]
목록 0: 3개 → 번호 [5 6 7]
목록 1: 8개 → 번호 [ 0  1  2  3  8  9 10 11]
목록 2: 1개 → 번호 [4]
점마다 들어간 목록: [1 1 1 1 2 0 0 0 1 1 1 1]


## 05-4. IVF 검색을 손으로 따라가기

In [4]:
q = np.array([[5.0, 5.0]], dtype=np.float32)          # 세 묶음 한가운데의 질문

def trace(nprobe, k=4):
    print(f"===== nprobe = {nprobe} =====")
    cd, cl = quantizer.search(q, nlist)                # 1단계: 질문 ↔ 중심점 nlist개
    for r, (c, dist) in enumerate(zip(cl[0], cd[0]), 1):
        mark = "← 뒤진다" if r <= nprobe else ""
        print(f"  1단계 {r}순위 목록 {c} (중심 {centroids[c]}) 거리 {dist:.2f} {mark}")
    cands = []
    for c in cl[0][:nprobe]:                           # 2단계: 고른 목록 안의 벡터들만
        for i in list_ids(c):
            dist = float(((x[i] - q[0]) ** 2).sum())
            cands.append((dist, int(i)))
            print(f"  2단계 목록 {c}의 {i}번 {x[i]} 거리 {dist:.0f}")
    cands.sort()                                       # 3단계: 거리 순 정렬 후 k개
    print("  손으로 구한 top-k:", [i for _, i in cands[:k]])
    ivf.nprobe = nprobe
    print("  faiss 결과      :", ivf.search(q, k)[1][0].tolist())
    print(f"  거리 계산: 중심 {nlist}번 + 후보 {len(cands)}번 = {nlist + len(cands)}번 (Flat은 {ivf.ntotal}번)")

trace(1)
trace(2)

===== nprobe = 1 =====
  1단계 1순위 목록 1 (중심 [1.5 5. ]) 거리 12.25 ← 뒤진다
  1단계 2순위 목록 2 (중심 [8. 8.]) 거리 18.00 
  1단계 3순위 목록 0 (중심 [8.666667 8.666667]) 거리 26.89 
  2단계 목록 1의 0번 [1. 1.] 거리 32
  2단계 목록 1의 1번 [1. 2.] 거리 25
  2단계 목록 1의 2번 [2. 1.] 거리 25
  2단계 목록 1의 3번 [2. 2.] 거리 18
  2단계 목록 1의 8번 [1. 8.] 거리 25
  2단계 목록 1의 9번 [1. 9.] 거리 32
  2단계 목록 1의 10번 [2. 8.] 거리 18
  2단계 목록 1의 11번 [2. 9.] 거리 25
  손으로 구한 top-k: [3, 10, 1, 2]
  faiss 결과      : [3, 10, 1, 2]
  거리 계산: 중심 3번 + 후보 8번 = 11번 (Flat은 12번)
===== nprobe = 2 =====
  1단계 1순위 목록 1 (중심 [1.5 5. ]) 거리 12.25 ← 뒤진다
  1단계 2순위 목록 2 (중심 [8. 8.]) 거리 18.00 ← 뒤진다
  1단계 3순위 목록 0 (중심 [8.666667 8.666667]) 거리 26.89 
  2단계 목록 1의 0번 [1. 1.] 거리 32
  2단계 목록 1의 1번 [1. 2.] 거리 25
  2단계 목록 1의 2번 [2. 1.] 거리 25
  2단계 목록 1의 3번 [2. 2.] 거리 18
  2단계 목록 1의 8번 [1. 8.] 거리 25
  2단계 목록 1의 9번 [1. 9.] 거리 32
  2단계 목록 1의 10번 [2. 8.] 거리 18
  2단계 목록 1의 11번 [2. 9.] 거리 25
  2단계 목록 2의 4번 [8. 8.] 거리 18
  손으로 구한 top-k: [3, 4, 10, 1]
  faiss 결과      : [3, 4, 10, 1]
  거리 계산: 중심 3번 + 후보 9

## 05-5. nprobe 바꾸기 — 인덱스 설정과 이번 검색만

In [5]:
for p in [1, 2, 3]:
    ivf.nprobe = p
    D, I = ivf.search(q, 4)
    print(f"nprobe={p} → {I.ravel()}  거리 {D.ravel()}")

params = faiss.SearchParametersIVF(nprobe=1)           # 이번 검색에만 적용
print("이번만 nprobe=1:", ivf.search(q, 4, params=params)[1], "| 인덱스 설정은 그대로:", ivf.nprobe)

nprobe=1 → [ 3 10  1  2]  거리 [18. 18. 25. 25.]
nprobe=2 → [ 3  4 10  1]  거리 [18. 18. 18. 25.]
nprobe=3 → [ 3  4 10  1]  거리 [18. 18. 18. 25.]
이번만 nprobe=1: [[ 3 10  1  2]] | 인덱스 설정은 그대로: 3


## 06-1. 2만 개로 실험 — nprobe별 정확도·거리 계산 수·시간

In [6]:
import faiss, time
import numpy as np

rng = np.random.default_rng(0)
d, n, k = 64, 20000, 10
centers = rng.normal(0, 10, (50, d)).astype("float32")          # 숨은 주제 50개
xb = (centers[rng.integers(0, 50, n)] + rng.normal(0, 1, (n, d))).astype("float32")   # 문서 2만 개
xq = (centers[rng.integers(0, 50, 100)] + rng.normal(0, 1, (100, d))).astype("float32")  # 질문 100개

flat = faiss.IndexFlatL2(d)
flat.add(xb)
t = time.perf_counter(); _, I_true = flat.search(xq, k); t_flat = time.perf_counter() - t

def recall(I):                     # 정답(전수 비교) k개 중 몇 개를 찾았나, 평균
    return np.mean([len(set(I[i]) & set(I_true[i])) / k for i in range(len(I))])

print(f"문서 {xb.shape}, 질문 {xq.shape}, Flat 검색 {t_flat*1000:.1f} ms (질문 100개)")

nlist = 100
ivf = faiss.IndexIVFFlat(faiss.IndexFlatL2(d), d, nlist)
ivf.train(xb)
ivf.add(xb)
sizes = [ivf.invlists.list_size(i) for i in range(nlist)]
print(f"목록 크기: 최소 {min(sizes)}, 최대 {max(sizes)}, 평균 {np.mean(sizes):.0f}")
for p in [1, 2, 4, 8, 16, 100]:
    ivf.nprobe = p
    faiss.cvar.indexIVF_stats.reset()                  # FAISS가 센 거리 계산 횟수 초기화
    t = time.perf_counter(); _, I = ivf.search(xq, k); el = time.perf_counter() - t
    ndis = faiss.cvar.indexIVF_stats.ndis / len(xq)    # 질문 1개당
    print(f"nprobe={p:<3} recall={recall(I):.3f}  질문당 거리계산 {ndis:7.0f}번 (Flat {n})  {el*1000:5.1f} ms")

문서 (20000, 64), 질문 (100, 64), Flat 검색 21.6 ms (질문 100개)
목록 크기: 최소 1, 최대 825, 평균 200
nprobe=1   recall=0.745  질문당 거리계산     291번 (Flat 20000)    0.9 ms
nprobe=2   recall=0.972  질문당 거리계산     489번 (Flat 20000)    0.9 ms
nprobe=4   recall=0.999  질문당 거리계산    1089번 (Flat 20000)    2.1 ms
nprobe=8   recall=1.000  질문당 거리계산    1901번 (Flat 20000)    2.4 ms
nprobe=16  recall=1.000  질문당 거리계산    3542번 (Flat 20000)    3.3 ms
nprobe=100 recall=1.000  질문당 거리계산   20000번 (Flat 20000)   32.8 ms


## 06-2. nlist 바꾸기 — 목록을 몇 개로 나눌까

In [7]:
for nl in [25, 100, 400]:
    ix = faiss.IndexIVFFlat(faiss.IndexFlatL2(d), d, nl)
    t = time.perf_counter(); ix.train(xb); t_train = time.perf_counter() - t
    ix.add(xb)
    for p in [1, 4]:
        ix.nprobe = p
        faiss.cvar.indexIVF_stats.reset()
        _, I = ix.search(xq, k)
        print(f"nlist={nl:<4} nprobe={p}  recall={recall(I):.3f}  질문당 거리계산 {faiss.cvar.indexIVF_stats.ndis/len(xq):6.0f}번  학습 {t_train:.2f}초")

nlist=25   nprobe=1  recall=0.997  질문당 거리계산   1799번  학습 0.08초
nlist=25   nprobe=4  recall=1.000  질문당 거리계산   8345번  학습 0.08초
nlist=100  nprobe=1  recall=0.745  질문당 거리계산    291번  학습 0.22초
nlist=100  nprobe=4  recall=0.999  질문당 거리계산   1089번  학습 0.22초
nlist=400  nprobe=1  recall=0.329  질문당 거리계산     72번  학습 0.56초
nlist=400  nprobe=4  recall=0.809  질문당 거리계산    266번  학습 0.56초


## 06-4. 운영 — 새 데이터가 몰려오면 목록이 기운다

In [8]:
new_topic = rng.normal(0, 10, (1, d)).astype("float32")           # 처음 보는 주제
xnew = (new_topic + rng.normal(0, 1, (5000, d))).astype("float32")
xq2 = (new_topic + rng.normal(0, 1, (50, d))).astype("float32")    # 새 주제에 대한 질문

def check(index, label):
    sizes = [index.invlists.list_size(i) for i in range(index.nlist)]
    full = faiss.IndexFlatL2(d); full.add(np.vstack([xb, xnew]))
    _, gt = full.search(xq2, k)
    print(f"{label}: 가장 큰 목록 {max(sizes)}개")
    for p in [4, 16]:
        index.nprobe = p
        faiss.cvar.indexIVF_stats.reset()
        _, I = index.search(xq2, k)
        r = np.mean([len(set(I[i]) & set(gt[i])) / k for i in range(len(I))])
        print(f"   nprobe={p:<2} 새 주제 질문 recall {r:.3f} | 질문당 거리계산 {faiss.cvar.indexIVF_stats.ndis/len(xq2):.0f}번")

ivf.add(xnew)                                          # 중심점은 그대로 둔 채 추가만
check(ivf, "추가만 한 인덱스")

re = faiss.IndexIVFFlat(faiss.IndexFlatL2(d), d, nlist)
re.train(np.vstack([xb, xnew]))                        # 새 분포로 다시 학습
re.add(np.vstack([xb, xnew]))
check(re, "다시 학습한 인덱스")

추가만 한 인덱스: 가장 큰 목록 3733개
   nprobe=4  새 주제 질문 recall 0.988 | 질문당 거리계산 5592번
   nprobe=16 새 주제 질문 recall 1.000 | 질문당 거리계산 8896번
다시 학습한 인덱스: 가장 큰 목록 820개
   nprobe=4  새 주제 질문 recall 0.416 | 질문당 거리계산 787번
   nprobe=16 새 주제 질문 recall 0.906 | 질문당 거리계산 3103번
